# Customer Segmentation with K-Means

**CMPE 255 — Assignment 1, Part 2**

This reproduction-ready experiment follows **CRISP-DM** to segment customers using the commonly studied Kaggle *Mall Customers* schema. It intentionally contains **no precomputed numerical findings**: all statistics, charts, model-selection evidence, profiles, and recommendations are generated only when a legitimate local CSV is supplied and the notebook is run.

> **Reproducibility note:** Run all cells from top to bottom. The notebook checks for `Mall_Customers.csv` before reading it and remains executable without the data.

## 1. Business Understanding

### Objective
Discover internally coherent customer groups that can support differentiated engagement, retention, and merchandising strategies. Clustering is exploratory—not a prediction of protected traits or individual behavior.

### Stakeholders and success criteria
- **Marketing:** interpretable groups that can inform campaign hypotheses.
- **Merchandising/customer experience:** understandable differences in income, age, and spending behavior.
- **Data science:** stable, reproducible preprocessing and defensible model selection.

Technical success is assessed through the elbow diagnostic, silhouette coefficient, cluster sizes, and interpretability across multiple candidate values of $K$. Business success would require a later controlled experiment measuring incremental outcomes; it cannot be established from this notebook alone.

### Ethical use
Segments should inform aggregate strategy, not deny services or enable discriminatory targeting. Human review, privacy controls, stability checks, and impact assessment are required before deployment.

## 2. Data Understanding

The expected file is `Mall_Customers.csv`, placed beside this notebook. The commonly used schema contains a customer identifier and attributes such as gender, age, annual income, and spending score. Exact columns are discovered at runtime rather than assumed.

**To execute with data:**
1. Obtain the dataset through its authorized Kaggle source and accept any applicable terms.
2. Save the unmodified CSV as `Mall_Customers.csv` in this directory.
3. Restart the kernel and choose **Run All**.

The CSV is deliberately not bundled in this repository.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
DATA_PATH = Path("Mall_Customers.csv")
IMAGE_DIR = Path("images")
IMAGE_DIR.mkdir(exist_ok=True)
plt.style.use("default")
pd.set_option("display.max_columns", 50)

DATA_AVAILABLE = DATA_PATH.is_file()
if DATA_AVAILABLE:
    print(f"Dataset found: {DATA_PATH.resolve()}")
else:
    print(
        "Dataset not found. Add an authorized copy named 'Mall_Customers.csv' "
        "beside this notebook, then restart the kernel and run all cells. "
        "Analytical cells will be skipped safely until then."
    )

In [ ]:
if DATA_AVAILABLE:
    raw_df = pd.read_csv(DATA_PATH)
    print("Shape:", raw_df.shape)
    display(raw_df.head())
    raw_df.info()
    display(raw_df.describe(include="all").T)
else:
    raw_df = None
    print("Data Understanding skipped: no CSV was loaded.")

## 3. Exploratory Data Analysis

The following cell produces runtime summaries and distributions without embedding results. Numeric histograms reveal shape and scale; categorical bar charts reveal balance. These are descriptive diagnostics, not evidence of causality.

In [ ]:
if DATA_AVAILABLE:
    numeric_columns = raw_df.select_dtypes(include=np.number).columns.tolist()
    categorical_columns = raw_df.select_dtypes(exclude=np.number).columns.tolist()
    print("Numeric columns:", numeric_columns)
    print("Categorical columns:", categorical_columns)

    if numeric_columns:
        axes = raw_df[numeric_columns].hist(figsize=(12, 8), bins=20, edgecolor="black")
        plt.suptitle("Numeric Feature Distributions", y=1.02)
        plt.tight_layout()
        plt.show()
    for column in categorical_columns:
        raw_df[column].value_counts(dropna=False).plot(kind="bar", title=f"Counts: {column}")
        plt.ylabel("Count")
        plt.tight_layout()
        plt.show()
else:
    print("EDA skipped: add the expected CSV and run all cells.")

## 4. Missing-Value Analysis

Missingness is reported as counts and percentages at runtime. Cleaning below uses median imputation for numeric fields and mode imputation for categorical fields, a transparent baseline that should be reconsidered if missingness is informative.

In [ ]:
if DATA_AVAILABLE:
    missing_report = pd.DataFrame({
        "missing_count": raw_df.isna().sum(),
        "missing_percent": raw_df.isna().mean().mul(100),
    }).sort_values("missing_percent", ascending=False)
    display(missing_report)
else:
    print("Missing-value analysis skipped: no CSV was loaded.")

## 5. Duplicate Analysis

Exact duplicate rows are counted. They are removed in the cleaning stage while retaining the first occurrence. Repeated customer identifiers are also reported separately because they can indicate a grain or integrity problem rather than exact duplication.

In [ ]:
if DATA_AVAILABLE:
    print("Exact duplicate rows:", int(raw_df.duplicated().sum()))
    id_candidates = [c for c in raw_df.columns if c.lower().replace("_", "") in {"customerid", "id"}]
    for column in id_candidates:
        print(f"Repeated non-null values in {column}:", int(raw_df[column].dropna().duplicated().sum()))
else:
    print("Duplicate analysis skipped: no CSV was loaded.")

## 6. Data Cleaning

Cleaning is performed on a copy: trim column names and string values, remove exact duplicates, coerce commonly numeric fields where possible, and impute remaining missing values. The raw frame remains unchanged for auditability.

In [ ]:
if DATA_AVAILABLE:
    clean_df = raw_df.copy()
    clean_df.columns = clean_df.columns.str.strip()
    object_columns = clean_df.select_dtypes(include="object").columns
    for column in object_columns:
        clean_df[column] = clean_df[column].map(lambda x: x.strip() if isinstance(x, str) else x)

    clean_df = clean_df.drop_duplicates().reset_index(drop=True)
    for column in clean_df.columns:
        normalized = column.lower().replace(" ", "").replace("_", "")
        if any(token in normalized for token in ("age", "income", "score")):
            clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")

    for column in clean_df.select_dtypes(include=np.number):
        clean_df[column] = clean_df[column].fillna(clean_df[column].median())
    for column in clean_df.select_dtypes(exclude=np.number):
        modes = clean_df[column].mode(dropna=True)
        if not modes.empty:
            clean_df[column] = clean_df[column].fillna(modes.iloc[0])

    print("Clean shape:", clean_df.shape)
    print("Remaining missing values:", int(clean_df.isna().sum().sum()))
else:
    clean_df = None
    print("Cleaning skipped: no CSV was loaded.")

## 7. Outlier Analysis

The IQR rule flags observations outside $Q_1-1.5\,IQR$ and $Q_3+1.5\,IQR$. Flags are reported and boxplots are shown, but observations are **not automatically deleted**: extreme customers may be valid and strategically meaningful. Standardization limits scale differences but is not itself outlier-robust.

In [ ]:
if DATA_AVAILABLE:
    analysis_numeric = clean_df.select_dtypes(include=np.number)
    q1 = analysis_numeric.quantile(0.25)
    q3 = analysis_numeric.quantile(0.75)
    iqr = q3 - q1
    outlier_mask = (analysis_numeric.lt(q1 - 1.5 * iqr) | analysis_numeric.gt(q3 + 1.5 * iqr))
    outlier_report = pd.DataFrame({
        "flagged_count": outlier_mask.sum(),
        "flagged_percent": outlier_mask.mean().mul(100),
    })
    display(outlier_report)
    if not analysis_numeric.empty:
        analysis_numeric.plot(kind="box", subplots=True, layout=(1, len(analysis_numeric)),
                              figsize=(max(10, 3 * len(analysis_numeric)), 4), sharex=False)
        plt.suptitle("Numeric Feature Outlier Diagnostics")
        plt.tight_layout()
        plt.show()
else:
    print("Outlier analysis skipped: no CSV was loaded.")

## 8. Feature Selection

Identifiers are excluded because their numeric ordering has no customer-behavior meaning. The baseline uses available numeric demographic/behavioral variables—preferentially age, income, and spending-score fields. This explicit, schema-aware rule avoids silently treating an ID as a feature. Categorical attributes remain available for post-hoc profiling rather than defining Euclidean distance.

In [ ]:
if DATA_AVAILABLE:
    numeric_candidates = clean_df.select_dtypes(include=np.number).columns.tolist()
    id_like = {c for c in clean_df.columns if c.lower().replace(" ", "").replace("_", "") in {"id", "customerid"}}
    preferred_tokens = ("age", "income", "spending", "score")
    preferred = [c for c in numeric_candidates if c not in id_like and any(t in c.lower() for t in preferred_tokens)]
    feature_columns = preferred or [c for c in numeric_candidates if c not in id_like]
    if len(feature_columns) < 2:
        raise ValueError("At least two non-identifier numeric features are required for clustering and PCA.")
    X = clean_df[feature_columns].copy()
    print("Selected features:", feature_columns)
else:
    feature_columns, X = [], None
    print("Feature selection skipped: no CSV was loaded.")

## 9. Feature Scaling with `StandardScaler`

K-Means uses Euclidean distance, so unequal units can dominate the solution. `StandardScaler` centers each selected feature and scales it to unit variance. The fitted scaler is retained for reproducibility and future inference.

In [ ]:
if DATA_AVAILABLE:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    scaled_df = pd.DataFrame(X_scaled, columns=feature_columns, index=X.index)
    display(scaled_df.head())
else:
    scaler, X_scaled = None, None
    print("Scaling skipped: no CSV was loaded.")

## 10–13. K-Means, Elbow Method, Silhouette Analysis, and Comparison of K

Candidate models use fixed initialization settings and random seed. Inertia supports the elbow diagnostic; mean silhouette score measures cohesion/separation (higher is better). No optimal $K$ is asserted in advance. For a reproducible baseline, the notebook selects the candidate with the highest observed silhouette score, while the analyst should also inspect the elbow, cluster balance, stability, and business interpretability.

Candidate $K$ values are constrained by sample size because silhouette analysis requires fewer clusters than observations.

In [ ]:
if DATA_AVAILABLE:
    if len(X_scaled) < 3:
        raise ValueError("At least three cleaned rows are required to compare clustering solutions.")
    k_values = list(range(2, min(10, len(X_scaled) - 1) + 1))
    model_rows = []
    candidate_models = {}
    for k in k_values:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = model.fit_predict(X_scaled)
        candidate_models[k] = model
        model_rows.append({
            "k": k,
            "inertia": model.inertia_,
            "silhouette_score": silhouette_score(X_scaled, labels),
            "smallest_cluster": int(pd.Series(labels).value_counts().min()),
            "largest_cluster": int(pd.Series(labels).value_counts().max()),
        })

    comparison_df = pd.DataFrame(model_rows)
    display(comparison_df)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(comparison_df["k"], comparison_df["inertia"], marker="o")
    axes[0].set(title="Elbow Method", xlabel="K", ylabel="Inertia")
    axes[1].plot(comparison_df["k"], comparison_df["silhouette_score"], marker="o")
    axes[1].set(title="Silhouette Analysis", xlabel="K", ylabel="Mean silhouette score")
    for ax in axes: ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(IMAGE_DIR / "model_selection_diagnostics.png", dpi=150, bbox_inches="tight")
    plt.show()

    selected_k = int(comparison_df.loc[comparison_df["silhouette_score"].idxmax(), "k"])
    final_model = candidate_models[selected_k]
    cluster_labels = final_model.labels_
    print("Runtime-selected K (maximum candidate silhouette):", selected_k)
else:
    comparison_df = None
    selected_k = final_model = cluster_labels = None
    print("Model comparison skipped: no CSV was loaded; no K or scores are claimed.")

## 14–15. PCA Dimensionality Reduction and Cluster Visualization

PCA projects standardized features into two dimensions **only for visualization**; K-Means remains fitted in the full selected feature space. Explained variance and coordinates are computed at runtime, not pre-reported. Proximity in the plot is an approximation and should not replace full-dimensional diagnostics.

In [ ]:
if DATA_AVAILABLE:
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"], index=X.index)
    pca_df["cluster"] = cluster_labels
    print("Explained variance ratio:", pca.explained_variance_ratio_)

    plt.figure(figsize=(9, 6))
    scatter = plt.scatter(pca_df["PC1"], pca_df["PC2"], c=pca_df["cluster"],
                          cmap="tab10", alpha=0.75, edgecolor="none")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.title(f"Customer Segments in PCA Space (runtime K={selected_k})")
    plt.colorbar(scatter, label="Cluster")
    plt.tight_layout()
    plt.savefig(IMAGE_DIR / "customer_segments_pca.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    pca = X_pca = pca_df = None
    print("PCA and visualization skipped: no CSV was loaded; no PCA result is claimed.")

## 16. Customer Segment Profiling

Profiles combine cluster size with numeric means and medians. Categorical distributions are shown when available. Cluster labels are arbitrary identifiers, not ordinal rankings.

In [ ]:
if DATA_AVAILABLE:
    segmented_df = clean_df.copy()
    segmented_df["Cluster"] = cluster_labels
    numeric_profile_columns = [c for c in feature_columns if c in segmented_df.columns]
    cluster_sizes = segmented_df["Cluster"].value_counts().sort_index().rename("size")
    mean_profiles = segmented_df.groupby("Cluster")[numeric_profile_columns].mean().add_suffix("_mean")
    median_profiles = segmented_df.groupby("Cluster")[numeric_profile_columns].median().add_suffix("_median")
    profile_table = pd.concat([cluster_sizes, mean_profiles, median_profiles], axis=1)
    display(profile_table)

    for column in segmented_df.select_dtypes(exclude=np.number).columns:
        print(f"Categorical distribution by cluster: {column}")
        display(pd.crosstab(segmented_df["Cluster"], segmented_df[column], normalize="index"))
else:
    segmented_df = profile_table = None
    print("Profiling skipped: no CSV was loaded; no cluster sizes are claimed.")

## 17. Interpretation of Each Cluster

Interpretations below are generated from observed standardized centroids. “Above” and “below” refer to the dataset mean in standard-deviation units; they are relative descriptions, not value judgments.

In [ ]:
if DATA_AVAILABLE:
    centroid_z = pd.DataFrame(final_model.cluster_centers_, columns=feature_columns)
    interpretation_rows = []
    for cluster_id, row in centroid_z.iterrows():
        descriptors = []
        for feature, value in row.items():
            if value >= 0.5:
                descriptors.append(f"above-average {feature}")
            elif value <= -0.5:
                descriptors.append(f"below-average {feature}")
            else:
                descriptors.append(f"near-average {feature}")
        interpretation_rows.append({"cluster": cluster_id, "data_driven_interpretation": "; ".join(descriptors)})
    interpretation_df = pd.DataFrame(interpretation_rows).set_index("cluster")
    display(interpretation_df)
else:
    interpretation_df = None
    print("Interpretation skipped: conclusions require execution on real data.")

## 18. Business Recommendations

Recommendations must be treated as **hypotheses derived after execution**, not established outcomes. Use the runtime profiles to:

- design differentiated messaging only where segment characteristics support it;
- prioritize retention research for groups with strong observed engagement;
- test value-oriented offers for lower-spending groups without assuming lack of willingness or ability;
- investigate high-income/low-spending patterns through voluntary research rather than intrusive targeting; and
- validate every intervention with randomized holdouts, incremental lift, cost, opt-out, fairness, and privacy metrics.

No segment-specific recommendation is asserted here because the original data was not executed. A stakeholder should name clusters only after reviewing the generated profiles and stability evidence.

## 19. Limitations

- The source is a small, cross-sectional teaching dataset and may not represent a broader population.
- Spending score provenance and construction may be unclear; it is not a direct causal outcome.
- K-Means assumes roughly spherical, similarly scaled clusters and is sensitive to outliers and initialization.
- Silhouette optimization is an internal criterion, not proof of business usefulness.
- PCA visualization loses information.
- Median/mode imputation can suppress uncertainty, while demographic variables introduce fairness concerns.
- Results may drift over time and do not establish causal effects or campaign lift.

## 20. Future Improvements

1. Validate schema, units, provenance, consent, and representativeness.
2. Quantify stability with repeated seeds, bootstrapping, and temporal holdouts.
3. Compare robust scaling and algorithms such as hierarchical clustering, Gaussian mixtures, and density-based methods.
4. Engineer recency-frequency-monetary and longitudinal behavioral features where legitimately available.
5. Use richer missingness analysis and sensitivity tests for outlier policies.
6. Assess fairness, privacy, actionability, and segment persistence before operational use.
7. Evaluate business value through preregistered experiments and incremental-lift measurement.

## 21. CRISP-DM Conclusion

This notebook connects **Business Understanding** to **Data Understanding**, **Data Preparation**, **Modeling**, and **Evaluation**, while treating **Deployment** as a governed future activity. It is intentionally reproduction-ready rather than result-bearing: when the authorized CSV is absent, cells explain what to do and safely skip computation. When supplied, all metrics, selected $K$, PCA output, visualizations, profiles, and interpretations are derived transparently at runtime.

A deployment decision should loop back through CRISP-DM: confirm stakeholder value, validate stability and fairness, document a scoring pipeline, monitor drift, and retire segments that cease to be useful.